# 价格合适 —— 第 7 周练习

## 练习目标

用 **QLoRA** 在轻量数据集 `ed-donner/items_prompts_lite` 上微调 `Llama-3.2-3B`，做商品价格预测；训练过程上报 **Weights & Biases**，并把适配器推到 Hugging Face Hub。

## 和本课第 7 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 4-bit 量化 | `BitsAndBytesConfig`（NF4） |
| LoRA | `LoraConfig` + `target_modules` |
| SFT | `SFTTrainer` / `SFTConfig` |
| 实验追踪 | `wandb.init` / `report_to="wandb"` |

## 怎么跑

1. Colab 开启 GPU；Secrets 配置 `HF_TOKEN`、`WANDB_API_KEY_2`
2. 按顺序运行安装 → 登录 → 加载数据 → 训练
3. 本笔记本把训练集截到 10,000 条以加快试验


In [ ]:
# ========== 依赖：钉死 bitsandbytes/trl，并拉取课程 util.py ==========

# 升级到与课程一致的版本（版本号字符串必须保持原样）
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1
# 下载 week7/util.py 到当前目录，供后续辅助函数使用
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py


In [ ]:
# ========== 安装 Weights & Biases 客户端 ==========

# -q 安静安装 wandb，供训练日志上报
!pip -q install wandb


In [2]:
# ========== 导入：Colab 登录、量化训练、W&B、画图 ==========

# 标准库 os：写 WANDB_* 环境变量
import os
# 标准库 re：正则辅助
import re
# 标准库 math：数值辅助
import math
# tqdm：进度条
from tqdm import tqdm
# Colab Secrets：读 HF_TOKEN / WANDB_API_KEY_2
from google.colab import userdata
# Hugging Face Hub 登录
from huggingface_hub import login
# PyTorch
import torch
# transformers 包本体
import transformers
# 因果 LM、分词器、TrainingArguments、种子、量化配置
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
# datasets：加载 Hub 数据
from datasets import load_dataset, Dataset, DatasetDict
# Weights & Biases
import wandb
# PEFT LoRA 配置
from peft import LoraConfig
# TRL 监督微调
from trl import SFTTrainer, SFTConfig
# 给 run 起时间戳名字
from datetime import datetime
# 画图（课程模板常保留）
import matplotlib.pyplot as plt


In [ ]:
# ========== 交互式 wandb 登录（relogin） ==========

# 在笔记本里触发 wandb 重新登录流程（命令字符串保持原样）
! wandb login --relogin


wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [4]:
# ========== 常量与超参数：lite 数据 + QLoRA + 训练追踪 ==========

# 底座模型 id（必须保持英文原样）
BASE_MODEL = "meta-llama/Llama-3.2-3B"
# W&B / 项目名
PROJECT_NAME = "price"
# 你的 Hugging Face 用户名（推送用）
HF_USER = "mrpeski"

# 使用轻量模式数据集
LITE_MODE = True

# 课程数据所有者与数据集名
ED_DATA_USER = "ed-donner"
ED_DATASET_NAME = f"{ED_DATA_USER}/items_prompts_lite"

# 本次 run 名：时间戳
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"

# 本地输出目录名 / Hub 仓库 id
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# ----- 总体超参数 -----

EPOCHS = 1
BATCH_SIZE = 32
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

# ----- QLoRA -----

QUANT_4_BIT = True
LORA_R = 32
# alpha 常取 2*r
LORA_ALPHA = LORA_R * 2
# 注意力投影层：挂 LoRA 的模块名
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
TARGET_MODULES = ATTENTION_LAYERS
LORA_DROPOUT = 0.1

# ----- 训练超参数 -----

LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

# GPU 算力主版本 >=8 → 可用 bf16
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# ----- 日志 / 验证规模 -----

VAL_SIZE = 500
LOG_STEPS = 5
SAVE_STEPS = 100
LOG_TO_WANDB = True


In [5]:
# ========== 登录 Hugging Face ==========

# 从 Colab Secrets 读取 HF_TOKEN（名称必须保持原样）
hf_token = userdata.get('HF_TOKEN')
# 登录 Hub，并写入 git credential
login(hf_token, add_to_git_credential=True)


## 实验追踪准备

下一格用 Colab Secret `WANDB_API_KEY_2` 登录 Weights & Biases，并把项目名等环境变量配好，便于训练曲线上报。


In [ ]:
# ========== 登录并配置 Weights & Biases ==========

# 注意：秘密名是 WANDB_API_KEY_2（与常见 WANDB_API_KEY 不同，保持原样）
wandb_api_key = userdata.get('WANDB_API_KEY_2')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# 挂到 PROJECT_NAME；不自动上传整模；不 watch 梯度
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"


In [ ]:
# ========== 加载 lite 数据集并截取验证集 ==========

# 按 ED_DATASET_NAME 从 Hub 拉取
dataset = load_dataset(ED_DATASET_NAME)
# 训练集（下一格还会再截到 10000）
train = dataset['train']
# 验证集只取前 VAL_SIZE 条，加快评估
val = dataset['val'].select(range(VAL_SIZE))
# 测试集保留（本笔记本后续未用，但保持原逻辑）
test = dataset['test']


In [8]:
# ========== 缩小训练集：只取前 10,000 条以加快试验 ==========

# 若你想用更大数据，可注释掉下一行
train = train.select(range(10000))


In [ ]:
# ========== 初始化 W&B run（指定 entity） ==========

# project / name / entity 字符串必须保持原样，才能写到作者的 W&B 团队空间
wandb.init(project=PROJECT_NAME, name=RUN_NAME, entity="mrpeski-olayinka-education")


## 现在加载分词器与模型

底座会以**量化**方式加载——把权重精度降到 4-bit（或 8-bit），显著省显存，这是 QLoRA 能在单卡上微调大模型的关键。


In [10]:
# ========== 选择量化：4-bit NF4 或 8-bit ==========

# QUANT_4_BIT=True 走 QLoRA 常用路径
if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    # Ampere+ 用 bfloat16，否则 float16
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )


In [ ]:
# ========== 加载分词器与量化底座 ==========

# 与 BASE_MODEL 匹配的 tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
# pad 用 eos；因果 LM 常用右侧 padding
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 按 quant_config 加载；device_map=auto 自动分配设备
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
# 生成配置的 pad_token_id 与 tokenizer 对齐
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# 打印显存占用（MB），确认量化生效
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")


In [14]:
# ========== LoRA 参数：只训练低秩适配器 ==========

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)


In [17]:
# ========== SFTConfig：本笔记本固定 fp16=False / bf16=True ==========

# 注意：这里硬编码 bf16=True（与上面 use_bf16 探测无关），需 Ampere+ GPU
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb",
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
# ========== 组装 SFTTrainer ==========

# 注入底座、train/val、LoRA 与训练参数
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters
)


In [ ]:
# ========== 开始微调并推送到 Hub ==========

# 启动训练循环（耗时取决于数据量与 GPU）
fine_tuning.train()

# 把微调后的适配器推到 Hugging Face；private=True 私有仓库
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
# 打印保存位置提示
print(f"Saved to the hub: {PROJECT_RUN_NAME}")


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss


In [ ]:
# ========== 结束 W&B run ==========

# 开关打开时收尾关闭，避免面板上一直显示 running
if LOG_TO_WANDB:
  wandb.finish()
